# Word embeddings (Word2Vec, GloVe, FastText) — runnable notebook
One focused concept, **5 runnable & visualizable examples** — each cell computes, plots, and asserts. Run-all safe.

In [ ]:
import numpy as np, matplotlib.pyplot as plt, math, itertools
np.random.seed(0)
def softmax(x, axis=-1):
    x=np.asarray(x, dtype=float); x=x-x.max(axis=axis, keepdims=True); e=np.exp(x); return e/e.sum(axis=axis, keepdims=True)
def sigmoid(x): return 1/(1+np.exp(-np.asarray(x, dtype=float)))
def normalize(v):
    v=np.asarray(v, dtype=float); n=np.linalg.norm(v); return v/(n if n else 1)
def edit_distance(a,b):
    D=np.zeros((len(a)+1,len(b)+1), dtype=int); D[:,0]=np.arange(len(a)+1); D[0,:]=np.arange(len(b)+1)
    for i in range(1,len(a)+1):
        for j in range(1,len(b)+1):
            D[i,j]=min(D[i-1,j]+1, D[i,j-1]+1, D[i-1,j-1]+(a[i-1]!=b[j-1]))
    return D
print('setup ok')

## Dense vectors encode distributional similarity through local context statistics
These examples move from co-occurrence counts to PPMI, cosine similarity, analogy geometry, and FastText-style subword sharing.

In [ ]:
words=['king','queen','man','woman']; C=np.array([[0,4,5,1],[4,0,1,5],[5,1,0,4],[1,5,4,0]],float)
plt.figure(figsize=(4,3)); plt.imshow(C,cmap='Blues'); plt.xticks(range(4),words); plt.yticks(range(4),words); plt.title('Toy co-occurrence matrix')
assert C[0,2]==5 and C[1,3]==5

In [ ]:
N=C.sum(); row=C.sum(1,keepdims=True); col=C.sum(0,keepdims=True); ppmi=np.maximum(np.log((C*N+1e-9)/(row@col+1e-9)),0)
plt.figure(figsize=(4,3)); plt.imshow(ppmi,cmap='magma'); plt.xticks(range(4),words); plt.yticks(range(4),words); plt.title('PPMI keeps surprising co-occurrences')
assert ppmi[0,2]>0

In [ ]:
vec={'king':np.array([1,1.0]),'queen':np.array([1,0.9]),'man':np.array([0.9,1.0]),'car':np.array([-1,-1.])}
sim=lambda a,b: float(normalize(vec[a])@normalize(vec[b])); vals=[sim('king',w) for w in ['queen','man','car']]
plt.figure(figsize=(5,3)); plt.bar(['queen','man','car'],vals); plt.title('Cosine similarity in embedding space')
assert vals[0]>vals[2]

In [ ]:
king=np.array([2.,1.]); man=np.array([1.,0.]); woman=np.array([1.,2.]); queen=king-man+woman
plt.figure(figsize=(4,4)); plt.scatter([king[0],man[0],woman[0],queen[0]],[king[1],man[1],woman[1],queen[1]]); [plt.text(*p,n) for n,p in {'king':king,'man':man,'woman':woman,'queen':queen}.items()]; plt.title('Analogy as vector offset')
assert np.allclose(queen,[2,3])

In [ ]:
def grams(w): return {w[i:i+3] for i in range(len(w)-2)}
G=[grams(w) for w in ['play','playing','played']]; overlap=[len(G[0]&g)/len(G[0]|g) for g in G]
plt.figure(figsize=(5,3)); plt.bar(['play','playing','played'],overlap); plt.title('FastText shares character n-grams')
assert overlap[1]>0 and overlap[2]>0